# Step 09: Function Calling with Auto-Invoke

This notebook demonstrates automatic function calling with plugins using FunctionChoiceBehavior.

In [1]:
import asyncio
import sys
from pathlib import Path
import logging

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
load_dotenv(dotenv_path, override=True)

True

In [2]:
from semantic_kernel import Kernel
from semantic_kernel.utils.logging import setup_logging
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from plugins.datetime import DateTimePlugin
from plugins.location import LocationPlugin
from plugins.weather import WeatherPlugin

In [3]:
# Enable logging (optional)
setup_logging()
logging.getLogger("semantic_kernel").setLevel(logging.DEBUG)
logging.getLogger("kernel").setLevel(logging.DEBUG)

In [4]:
# Initialize kernel and plugins
kernel = Kernel()
azure_chat_completion = AzureChatCompletion(instruction_role="system")
kernel.add_service(azure_chat_completion)

kernel.add_plugin(LocationPlugin(), plugin_name="location_plugin")
kernel.add_plugin(DateTimePlugin(), plugin_name="time_plugin")
kernel.add_plugin(WeatherPlugin(), plugin_name="weather_plugin")

KernelPlugin(name='weather_plugin', description=None, functions={'GetCurrentWeather': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='GetCurrentWeather', plugin_name='weather_plugin', description='Get current weather data based on latitude and longitude.', parameters=[KernelParameterMetadata(name='latitude', description='Latitude of the location', default_value=None, type_='float', is_required=True, type_object=<class 'float'>, schema_data={'type': 'number', 'description': 'Latitude of the location'}, include_in_function_choices=True), KernelParameterMetadata(name='longitude', description='Longitude of the location', default_value=None, type_='float', is_required=True, type_object=<class 'float'>, schema_data={'type': 'number', 'description': 'Longitude of the location'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='', default_value=None, type_='str', is_required=True, typ

In [5]:
# Configure automatic function calling
execution_settings = AzureChatPromptExecutionSettings(
    function_choice_behavior=FunctionChoiceBehavior.Auto(
        auto_invoke=True,
        filters={
            "included_plugins": [
                "time_plugin",
                "location_plugin",
                "weather_plugin",
            ]
        },
    )
)

history = ChatHistory()

In [6]:
# Interactive chat function
async def chat(user_input: str):
    history.add_user_message(user_input)
    result = await azure_chat_completion.get_chat_message_content(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
    )
    print("Assistant > " + str(result))
    history.add_message(result)
    return result

In [7]:
# Example: Ask about weather
await chat("What's the weather like?")

[2026-06-18 15:35:30 - semantic_kernel.connectors.ai.open_ai.services.open_ai_handler:231 - INFO] OpenAI usage: CompletionUsage(completion_tokens=15, prompt_tokens=185, total_tokens=200, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 60, 'engine_ttft_ms': 125, 'engine_ttlt_ms': 1025, 'pre_inference_ms': 80, 'service_tbt_ms': 60, 'service_ttft_ms': 591, 'service_ttlt_ms': 1483, 'total_duration_ms': 1412, 'user_visible_ttft_ms': 511})
[2026-06-18 15:35:30 - semantic_kernel.connectors.ai.chat_completion_client_base:149 - INFO] processing 1 tool calls in parallel.
[2026-06-18 15:35:30 - semantic_kernel.kernel:426 - INFO] Calling location_plugin-GetLocationInfo function with args: {}
[2026-06-18 15:35:30 - semantic_kernel.functions.kernel_function:19 - INFO] Function location_plugi

Assistant > It's currently 17.7°C in Athlone, Leinster, Ireland, with a wind speed of 15.1 m/s.


ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-Ds8849IqOfD6zre5e65ZopeKDauPN', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="It's currently 17.7°C in Athlone, Leinster, Ireland, with a wind speed of 15.1 m/s.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1781793332, model='gpt-4o-2024-11-20', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_49e2bef596', usage=CompletionUsage(completion_tokens=31, prompt_tokens=297, total_tokens=328, completion_tokens_detai